# 08 - Equal-budget settings search

Offers CWGAN, CWGAN-GP and CA-CWGAN the same eight (batch, learning rate, noise
dimension) candidates at a 2,000-update proxy budget and scores each by the macro
F1 of a classifier trained on synthetic rows only (Table 5). GPU. Run once per
dataset. See `PIPELINE.md`, step 8.

In [ ]:
# Colab: upload the converted dataset CSV written by notebook 01 (not df_train_*.csv).
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
%pip install -q lightgbm

In [ ]:
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightgbm as lgb
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import f1_score

try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."
CHECKPOINT_DIR = "."      # trial checkpoints and the search record; a Drive path survives a disconnect
MAX_MINUTES = 150         # stop cleanly; re-run unchanged to continue

ARMS = ["cwgan", "cwgangp", "ecwgan"]
N_ITER = 8                # candidates drawn once and offered to every arm
SEARCH_SPACE = dict(
    batch     = [128, 250, 256, 500],
    lr        = [1e-4, 2e-4, 5e-4],
    noise_dim = [32, 64, 128],
)
SEARCH_UPDATES      = 2000    # proxy budget, identical for every arm
VAL_FRACTION        = 0.20    # validation rows carved out of the training rows
TUNE_MAX_ROWS       = 20000   # stratified cap on the search's training rows
TUNE_POOL_PER_CLASS = 2000    # synthetic rows per class used to score a trial

SEED           = 42
TEST_SIZE      = 0.20
MIN_CLASS_SIZE = 10
MAX_DISCRETE_LEVELS = 12
SAVE_EVERY     = 1000

# Generator constants, identical to notebooks 02 and 03.
CRITIC_ITER = 5
CLIP_VALUE  = 0.01        # cwgan
LAMBDA_GP   = 10          # cwgangp and ecwgan
N_MODES     = 10
TAU         = 0.0
GUMBEL_TAU  = 0.2
PAC         = 10
KNN_K       = 5
KNN_MIN_OWN = 0.4
MAX_DROP    = 0.5
MAX_ROUNDS  = 8
KNN_BACKEND = "auto"
KNN_CHUNK   = 1024
GEN_CHUNK   = 131072

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > T4 GPU.")
DEVICE = "cuda"

## Data (notebook 01 steps 1-6)

In [ ]:
PRESETS = {
    "diabetes":   dict(file="diabetes_012_health_indicators_BRFSS2015.csv",
                       target="Diabetes_012", drop=[]),
    "yeast":      dict(file="yeast.csv",       target="Target", drop=["Sequence_Name"]),
    "ecoli":      dict(file="ecoli.csv",       target="Target", drop=["Sequence_Name"]),
    "pageblocks": dict(file="page-blocks.csv", target="Target", drop=[]),
}


def load_clean(dataset):
    """Load, drop identifiers and duplicates, remove tiny classes, encode the target."""
    cfg = PRESETS[dataset]
    target = cfg["target"]
    df = pd.read_csv(os.path.join(DATA_DIR, cfg["file"]))
    df = df.drop(columns=[c for c in cfg["drop"] if c in df.columns])
    df = df.drop_duplicates().reset_index(drop=True).dropna().reset_index(drop=True)
    counts = df[target].value_counts()
    tiny = counts[counts < MIN_CLASS_SIZE]
    if len(tiny):
        df = df[~df[target].isin(tiny.index)].reset_index(drop=True)
    labels = sorted(df[target].unique())
    df[target] = df[target].map({lab: i for i, lab in enumerate(labels)}).astype(int)
    return df, target


def split(clean, target, seed):
    """Stratified 80/20, then Min-Max fitted on the training rows only."""
    tr, te = train_test_split(clean, test_size=TEST_SIZE, stratify=clean[target],
                              random_state=seed, shuffle=True)
    tr, te = tr.reset_index(drop=True), te.reset_index(drop=True)
    feats = [c for c in clean.columns if c != target]
    sc = MinMaxScaler().fit(tr[feats])
    out = []
    for part in (tr, te):
        d = pd.DataFrame(sc.transform(part[feats]), columns=feats)
        d[target] = part[target].values
        out.append(d)
    return out[0], out[1]


def stratified_split(df, frac, seed=SEED):
    va = pd.concat([g.sample(max(1, int(round(len(g) * frac))), random_state=seed)
                    for _, g in df.groupby(TARGET_COL)])
    return df.drop(index=va.index).reset_index(drop=True), va.reset_index(drop=True)


def stratified_cap(df, n_total, seed=SEED):
    if len(df) <= n_total:
        return df
    frac = n_total / len(df)
    return pd.concat([g.sample(max(2, int(round(len(g) * frac))), random_state=seed)
                      for _, g in df.groupby(TARGET_COL)]).reset_index(drop=True)


CLEAN, TARGET_COL = load_clean(DATASET)
print("loaded:", CLEAN.shape, "| class counts:",
      CLEAN[TARGET_COL].value_counts().sort_index().tolist())

In [ ]:
def setup(df_train):
    """Globals the generator code reads, computed from the rows being trained on."""
    global FEATURES, CLASSES, input_dim, cond_dim, ohe, SCHEMA
    global DISCRETE_COLS, CONTINUOUS_COLS, LEVELS
    FEATURES = [c for c in df_train.columns if c != TARGET_COL]
    CLASSES = np.sort(df_train[TARGET_COL].unique())
    v = df_train[FEATURES].to_numpy(dtype=float)
    assert v.min() >= -1e-6 and v.max() <= 1 + 1e-6, "training features are not in [0, 1]"
    input_dim, cond_dim = len(FEATURES), len(CLASSES)
    ohe = OneHotEncoder(sparse_output=False, categories=[CLASSES])
    ohe.fit(CLASSES.reshape(-1, 1))
    SCHEMA = build_schema(df_train)
    DISCRETE_COLS, CONTINUOUS_COLS, LEVELS = [], [], {}
    for col in FEATURES:
        lv = np.unique(df_train[col].to_numpy(dtype=float))
        if len(lv) <= MAX_DISCRETE_LEVELS:
            DISCRETE_COLS.append(col); LEVELS[col] = np.sort(lv)
        else:
            CONTINUOUS_COLS.append(col)

## CWGAN / CWGAN-GP (as notebook 02; the two differ only in the Lipschitz mechanism)

In [ ]:
class CWGAN_Generator(nn.Module):
    def __init__(self, noise_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, input_dim), nn.Sigmoid(),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class CWGAN_Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + cond_dim, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))


def cwgan_gradient_penalty(D, real_x, fake_x, y):
    alpha = torch.rand(real_x.size(0), 1, device=DEVICE).expand_as(real_x)
    mixed = (alpha * real_x + (1 - alpha) * fake_x).requires_grad_(True)
    scores = D(mixed, y)
    grads = torch.autograd.grad(outputs=scores, inputs=mixed,
                                grad_outputs=torch.ones_like(scores),
                                create_graph=True, retain_graph=True, only_inputs=True)[0]
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


def build_schema(df_reference):
    schema = {"discrete": {}, "continuous": {}}
    for col in [c for c in df_reference.columns if c != TARGET_COL]:
        levels = np.unique(df_reference[col].dropna().to_numpy(dtype=float))
        if len(levels) <= MAX_DISCRETE_LEVELS:
            schema["discrete"][col] = np.sort(levels)
        else:
            schema["continuous"][col] = (float(df_reference[col].min()),
                                         float(df_reference[col].max()))
    return schema


def apply_schema(df_synth, schema):
    out = df_synth.copy()
    for col, levels in schema["discrete"].items():
        v = out[col].to_numpy(dtype=float)
        out[col] = levels[np.abs(v[:, None] - levels[None, :]).argmin(axis=1)]
    for col, (lo, hi) in schema["continuous"].items():
        out[col] = out[col].clip(lo, hi)
    return out

## CA-CWGAN (as notebook 03)

In [ ]:
class ColumnTransform:
    def fit(self, df):
        self.spans = [(c, "discrete", len(LEVELS[c])) for c in DISCRETE_COLS]
        self.gmm, self.active = {}, {}
        for c in CONTINUOUS_COLS:
            v = df[c].to_numpy(dtype=float).reshape(-1, 1)
            g = BayesianGaussianMixture(
                n_components=min(N_MODES, max(1, len(np.unique(v)))),
                weight_concentration_prior_type="dirichlet_process",
                weight_concentration_prior=1e-3, max_iter=100, random_state=SEED)
            g.fit(v)
            act = g.weights_ > 1e-3
            if not act.any():
                act[np.argmax(g.weights_)] = True
            self.gmm[c], self.active[c] = g, act
            self.spans.append((c, "vgm", 1 + int(act.sum())))
        self.dim = sum(w for _, _, w in self.spans)
        return self

    def transform(self, df):
        blocks = []
        for col, kind, width in self.spans:
            v = df[col].to_numpy(dtype=float)
            if kind == "discrete":
                lv = LEVELS[col]
                idx = np.abs(v[:, None] - lv[None, :]).argmin(axis=1)
                b = np.zeros((len(v), width), dtype=np.float32)
                b[np.arange(len(v)), idx] = 1.0
            else:
                g, act = self.gmm[col], self.active[col]
                mu = g.means_.reshape(-1)[act]
                sd = np.sqrt(g.covariances_.reshape(-1))[act]
                mode = g.predict_proba(v.reshape(-1, 1))[:, act].argmax(axis=1)
                alpha = np.clip((v - mu[mode]) / (4 * np.maximum(sd[mode], 1e-6)), -0.99, 0.99)
                b = np.zeros((len(v), width), dtype=np.float32)
                b[:, 0] = alpha
                b[np.arange(len(v)), 1 + mode] = 1.0
            blocks.append(b)
        return np.concatenate(blocks, axis=1).astype(np.float32)

    def inverse(self, M):
        out, pos = {}, 0
        for col, kind, width in self.spans:
            blk = M[:, pos:pos + width]; pos += width
            if kind == "discrete":
                out[col] = LEVELS[col][blk.argmax(axis=1)]
            else:
                g, act = self.gmm[col], self.active[col]
                mu = g.means_.reshape(-1)[act]
                sd = np.sqrt(g.covariances_.reshape(-1))[act]
                mode = blk[:, 1:].argmax(axis=1)
                out[col] = np.clip(np.clip(blk[:, 0], -1, 1) * 4 * sd[mode] + mu[mode], 0, 1)
        return pd.DataFrame(out)[FEATURES]

    def heads(self):
        plan, pos = [], 0
        for col, kind, width in self.spans:
            if kind == "discrete":
                plan.append(("softmax", pos, width))
            else:
                plan.append(("tanh", pos, 1)); plan.append(("softmax", pos + 1, width - 1))
            pos += width
        return plan


PACK = PAC


class ECWGAN_Generator(nn.Module):
    def __init__(self, noise_dim, out_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, out_dim),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class ECWGAN_Critic(nn.Module):
    def __init__(self, in_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear((in_dim + cond_dim) * PACK, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        j = torch.cat([x, y], dim=1)
        return self.net(j.reshape(-1, j.size(1) * PACK))


def activate(raw, hard=False):
    parts = []
    for kind, start, width in CT.heads():
        blk = raw[:, start:start + width]
        if kind == "tanh":
            parts.append(torch.tanh(blk))
        else:
            parts.append(F.gumbel_softmax(blk, tau=GUMBEL_TAU, hard=hard, dim=1))
    return torch.cat(parts, dim=1)


def ecwgan_gradient_penalty(D, real, fake, cond):
    a = torch.rand(real.size(0) // PACK, 1, 1, device=DEVICE)
    a = a.repeat(1, PACK, real.size(1)).reshape(-1, real.size(1))
    mixed = (a * real + (1 - a) * fake).requires_grad_(True)
    scores = D(mixed, cond)
    g = torch.autograd.grad(scores, mixed, torch.ones_like(scores),
                            create_graph=True, retain_graph=True, only_inputs=True)[0]
    return ((g.reshape(-1, PACK * real.size(1)).norm(2, dim=1) - 1) ** 2).mean() * LAMBDA_GP


def ecwgan_prepare(df):
    global CT
    CT = ColumnTransform().fit(df)
    X_enc = CT.transform(df)
    labels = df[TARGET_COL].to_numpy()
    classes = np.sort(np.unique(labels))
    by_class = {int(k): np.flatnonzero(labels == k) for k in classes}
    counts = np.array([len(by_class[int(k)]) for k in classes], dtype=float)
    w = np.power(counts, TAU)
    return {"ct": CT, "X": torch.tensor(X_enc, device=DEVICE), "classes": classes,
            "cond_dim": len(classes), "by_class": by_class, "cond_p": w / w.sum(),
            "eye": torch.eye(len(classes), device=DEVICE)}

## Checkpointing

In [ ]:
if str(CHECKPOINT_DIR).startswith("/content/drive"):
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SEARCH_FILE   = os.path.join(CHECKPOINT_DIR, f"search_trials_{DATASET}.csv")
SELECTED_FILE = os.path.join(CHECKPOINT_DIR, f"search_selected_{DATASET}.json")


def ckpt_path(stem):
    return os.path.join(CHECKPOINT_DIR, f"ckpt_{stem}.pt")


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def rng_restore(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim=None):
    payload = {"dataset": DATASET, "stem": stem, "cfg": cfg, "updates_done": done,
               "history": history, "ct_dim": ct_dim,
               "G": G.state_dict(), "D": D.state_dict(),
               "opt_G": oG.state_dict(), "opt_D": oD.state_dict(), "rng": rng_state()}
    tmp = ckpt_path(stem) + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, ckpt_path(stem))


def load_ckpt(stem, cfg, ct_dim=None):
    path = ckpt_path(stem)
    if not os.path.isfile(path):
        return None
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    same = (ck.get("dataset") == DATASET and ck.get("stem") == stem
            and ck.get("cfg") == cfg and ck.get("ct_dim") == ct_dim)
    return ck if same else None


def resume_or_start(stem, cfg, G, D, oG, oD, seed, ct_dim=None):
    ck = load_ckpt(stem, cfg, ct_dim)
    if ck is None:
        set_seed(seed)
        return 0, []
    G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
    oG.load_state_dict(ck["opt_G"]); oD.load_state_dict(ck["opt_D"])
    rng_restore(ck["rng"])
    return ck["updates_done"], list(ck["history"])

## Training and pools

In [ ]:
def train_cwgan(df, cfg, stem, max_minutes, seed, use_gp):
    X_t = torch.tensor(df[FEATURES].to_numpy(dtype="float32"), device=DEVICE)
    y_t = torch.tensor(ohe.transform(df[[TARGET_COL]].to_numpy()).astype("float32"),
                       device=DEVICE)
    batch = int(cfg["batch"])
    log_every = max(1, cfg["updates"] // 200)
    G = CWGAN_Generator(cfg["noise_dim"]).to(DEVICE)
    D = CWGAN_Critic().to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
    oD = torch.optim.Adam(D.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
    done, history = resume_or_start(stem, cfg, G, D, oG, oD, seed)

    deadline = time.time() + max_minutes * 60
    for u in range(done, cfg["updates"]):
        for _ in range(CRITIC_ITER):
            idx = np.random.randint(0, len(X_t), batch)
            real_x, real_y = X_t[idx], y_t[idx]
            z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
            fake_x = G(z, real_y)
            loss_D = -(D(real_x, real_y).mean() - D(fake_x.detach(), real_y).mean())
            if use_gp:
                loss_D = loss_D + LAMBDA_GP * cwgan_gradient_penalty(
                    D, real_x, fake_x.detach(), real_y)
            oD.zero_grad(set_to_none=True); loss_D.backward(); oD.step()
            if not use_gp:
                with torch.no_grad():
                    for p in D.parameters():
                        p.clamp_(-CLIP_VALUE, CLIP_VALUE)
        z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
        loss_G = -D(G(z, real_y), real_y).mean()
        oG.zero_grad(set_to_none=True); loss_G.backward(); oG.step()

        done = u + 1
        if done % log_every == 0 or done == cfg["updates"]:
            history.append((done, float(loss_D.detach()), float(loss_G.detach())))
        if done % SAVE_EVERY == 0:
            save_ckpt(stem, cfg, G, D, oG, oD, done, history)
        if time.time() > deadline:
            break
    save_ckpt(stem, cfg, G, D, oG, oD, done, history)
    return G, history, done


def generate_pool(G, cfg, per_class, chunk=131072):
    G.eval()
    parts = []
    with torch.no_grad():
        for cls in CLASSES:
            cond = ohe.transform(np.array([[cls]])).astype("float32")
            made = 0
            while made < per_class:
                n = min(chunk, per_class - made)
                z = torch.randn(n, cfg["noise_dim"], device=DEVICE)
                y = torch.tensor(np.repeat(cond, n, axis=0), device=DEVICE)
                part = pd.DataFrame(G(z, y).cpu().numpy(), columns=FEATURES)
                part[TARGET_COL] = cls
                parts.append(part)
                made += n
    G.train()
    return pd.concat(parts, ignore_index=True)[[*FEATURES, TARGET_COL]]


def train_ecwgan(ctx, cfg, stem, max_minutes, seed):
    X_t, eye, by_class = ctx["X"], ctx["eye"], ctx["by_class"]
    classes, cond_dim, cond_p = ctx["classes"], ctx["cond_dim"], ctx["cond_p"]
    ct_dim = ctx["ct"].dim
    batch = max(PACK, (int(cfg["batch"]) // PACK) * PACK)
    log_every = max(1, cfg["updates"] // 200)
    G = ECWGAN_Generator(cfg["noise_dim"], ct_dim, cond_dim).to(DEVICE)
    D = ECWGAN_Critic(ct_dim, cond_dim).to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
    oD = torch.optim.Adam(D.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))

    def sample_cond(n):
        drawn = np.random.choice(cond_dim, size=n, p=cond_p)
        rows = np.empty(n, dtype=np.int64)
        for ci in range(cond_dim):
            m = drawn == ci
            k = int(m.sum())
            if k:
                pool_idx = by_class[int(classes[ci])]
                rows[m] = pool_idx[np.random.randint(0, len(pool_idx), k)]
        return eye[torch.from_numpy(drawn).long().to(DEVICE)], rows

    done, history = resume_or_start(stem, cfg, G, D, oG, oD, seed, ct_dim)
    deadline = time.time() + max_minutes * 60
    for u in range(done, cfg["updates"]):
        dl = 0.0
        for _ in range(CRITIC_ITER):
            cond, rows = sample_cond(batch)
            real = X_t[torch.from_numpy(rows).long().to(DEVICE)]
            z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
            fake = activate(G(z, cond))
            loss_D = D(fake.detach(), cond).mean() - D(real, cond).mean()
            loss_D = loss_D + ecwgan_gradient_penalty(D, real, fake.detach(), cond)
            oD.zero_grad(set_to_none=True); loss_D.backward(); oD.step()
            dl += float(loss_D.detach())
        cond, _ = sample_cond(batch)
        z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
        loss_G = -D(activate(G(z, cond)), cond).mean()
        oG.zero_grad(set_to_none=True); loss_G.backward(); oG.step()

        done = u + 1
        if done % log_every == 0 or done == cfg["updates"]:
            history.append((done, dl / CRITIC_ITER, float(loss_G.detach())))
        if done % SAVE_EVERY == 0:
            save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim)
        if time.time() > deadline:
            break
    save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim)
    return G, history, done


def knn_indices(reference, query, k):
    big = len(query) * len(reference) > 2e8
    use_torch = KNN_BACKEND == "torch" or (
        KNN_BACKEND == "auto" and torch.cuda.is_available() and big)
    if not use_torch:
        nn_ = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(reference)
        return nn_.kneighbors(query, return_distance=False)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    R = torch.as_tensor(np.ascontiguousarray(reference), dtype=torch.float32, device=dev)
    out = np.empty((len(query), k), dtype=np.int64)
    s, chunk = 0, KNN_CHUNK
    while s < len(query):
        try:
            Q = torch.as_tensor(np.ascontiguousarray(query[s:s + chunk]),
                                dtype=torch.float32, device=dev)
            out[s:s + chunk] = torch.cdist(Q, R).topk(k, dim=1, largest=False).indices.cpu().numpy()
            s += chunk
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if chunk <= 64:
                raise
            chunk //= 2
    del R
    if dev == "cuda":
        torch.cuda.empty_cache()
    return out


def build_pool(G, ctx, cfg, ref_df, per_class):
    """CA-CWGAN pool with E5 (alignment + k-NN vote), as notebook 03."""
    classes, eye, ct = ctx["classes"], ctx["eye"], ctx["ct"]
    step = max(PACK, (GEN_CHUNK // PACK) * PACK)
    ref_X = ref_df[FEATURES].to_numpy(dtype=np.float32)
    ref_y = ref_df[TARGET_COL].to_numpy()
    ranges = {c: (g[FEATURES].min(), g[FEATURES].max()) for c, g in ref_df.groupby(TARGET_COL)}

    def generate(cls, n):
        idx = int(np.flatnonzero(classes == cls)[0])
        chunks, made = [], 0
        with torch.no_grad():
            while made < n:
                size = max(min(step, n - made), PACK)
                z = torch.randn(size, cfg["noise_dim"], device=DEVICE)
                cond = eye[torch.full((size,), idx, dtype=torch.long, device=DEVICE)]
                chunks.append(activate(G(z, cond), hard=True).cpu().numpy())
                made += size
        return ct.inverse(np.concatenate(chunks)[:n])

    def align(df_part, cls):
        out = df_part.copy()
        lo, hi = ranges[cls]
        out[FEATURES] = out[FEATURES].clip(lo, hi, axis=1)
        for col in FEATURES:
            if col in LEVELS:
                lv = LEVELS[col]
                v = out[col].to_numpy(dtype=float)
                pos = np.clip(np.searchsorted(lv, v), 1, len(lv) - 1)
                left, right = lv[pos - 1], lv[pos]
                out[col] = np.where(np.abs(v - left) <= np.abs(v - right), left, right)
        return out

    def denoise(df_part, cls):
        k = min(KNN_K, len(ref_df) - 1)
        idx = knn_indices(ref_X, df_part[FEATURES].to_numpy(dtype=np.float32), k)
        own = (ref_y[idx] == cls).mean(axis=1)
        keep = own >= KNN_MIN_OWN
        cap = int(len(df_part) * MAX_DROP)
        if (~keep).sum() > cap:
            order = np.argsort(-own)
            keep = np.zeros(len(df_part), bool)
            keep[order[:len(df_part) - cap]] = True
        return df_part.loc[keep].reset_index(drop=True)

    G.eval()
    parts = []
    for cls in classes:
        kept, have, rounds = [], 0, 0
        while have < per_class and rounds < MAX_ROUNDS:
            part = generate(cls, int(np.ceil((per_class - have) / (1 - MAX_DROP))))
            part[TARGET_COL] = cls
            part = denoise(align(part, cls), cls)
            kept.append(part)
            have += len(part)
            rounds += 1
        part = pd.concat(kept, ignore_index=True).iloc[:per_class].reset_index(drop=True)
        if len(part) < per_class:
            raise RuntimeError(f"class {cls}: only {len(part):,} of {per_class:,} rows survived")
        parts.append(part)
    G.train()
    return pd.concat(parts, ignore_index=True)[ref_df.columns]


def cwgan_pool(df_train, cfg, per_class, seed, stem, max_minutes, use_gp):
    set_seed(seed)
    G, history, done = train_cwgan(df_train, cfg, stem, max_minutes, seed, use_gp)
    if done < cfg["updates"]:
        return None, history, False
    set_seed(seed + 1000)
    return apply_schema(generate_pool(G, cfg, per_class), SCHEMA), history, True


def ecwgan_pool(df_train, cfg, per_class, seed, stem, max_minutes):
    set_seed(seed)
    ctx = ecwgan_prepare(df_train)
    G, history, done = train_ecwgan(ctx, cfg, stem, max_minutes, seed)
    if done < cfg["updates"]:
        return None, history, False
    set_seed(seed + 1000)
    return build_pool(G, ctx, cfg, df_train, per_class), history, True


POOL = {
    "cwgan":   lambda *a, **k: cwgan_pool(*a, use_gp=False, **k),
    "cwgangp": lambda *a, **k: cwgan_pool(*a, use_gp=True, **k),
    "ecwgan":  ecwgan_pool,
}

## Search

In [ ]:
def trial_key(c):
    return f"b{c['batch']}_lr{c['lr']:.0e}_z{c['noise_dim']}"


def score_trial(pool, val):
    """Macro F1 of LightGBM trained on synthetic rows only, scored on real validation rows."""
    m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.08, num_leaves=31,
                           random_state=SEED, verbosity=-1, n_jobs=-1)
    m.fit(pool[FEATURES], pool[TARGET_COL])
    return float(f1_score(val[TARGET_COL], m.predict(val[FEATURES]),
                          average="macro", zero_division=0))


def run_search():
    if os.path.isfile(SELECTED_FILE):
        print(os.path.basename(SELECTED_FILE), "exists - delete it to search again")
        return
    df_train, _ = split(CLEAN, TARGET_COL, SEED)      # the hold-out is never used here
    tr, va = stratified_split(df_train, VAL_FRACTION, SEED)
    tr = stratified_cap(tr, TUNE_MAX_ROWS, SEED)
    setup(tr)
    per_class = min(TUNE_POOL_PER_CLASS, int(tr[TARGET_COL].value_counts().max()) * 4)
    ref = score_trial(tr, va)
    print(f"search on {len(tr):,} rows, validation {len(va):,} rows, "
          f"real-data reference macro F1 {ref:.4f}")

    done = set()
    if os.path.isfile(SEARCH_FILE):
        prev = pd.read_csv(SEARCH_FILE)
        done = set(zip(prev["arm"], prev["trial"]))

    cands = list(ParameterSampler(SEARCH_SPACE, n_iter=N_ITER, random_state=SEED))
    deadline = time.time() + MAX_MINUTES * 60
    for arm in ARMS:
        for i, c in enumerate(cands, 1):
            cfg = dict(updates=SEARCH_UPDATES, **{k: c[k] for k in SEARCH_SPACE})
            key = trial_key(cfg)
            if (arm, key) in done:
                continue
            if time.time() > deadline:
                print("time limit - re-run unchanged to continue")
                return
            t0 = time.time()
            stem = f"search_{DATASET}_{arm}_{key}"
            pool, hist, ok = POOL[arm](tr, cfg, per_class, SEED, stem,
                                       max_minutes=(deadline - time.time()) / 60)
            if not ok:
                print(f"{arm} {key}: time limit mid-training - re-run to continue")
                return
            if os.path.isfile(ckpt_path(stem)):
                os.remove(ckpt_path(stem))
            f1 = score_trial(pool, va)
            row = dict(dataset=DATASET, arm=arm, trial=key, **{k: cfg[k] for k in SEARCH_SPACE},
                       search_updates=SEARCH_UPDATES, tstr_f1=round(f1, 6),
                       real_reference_f1=round(ref, 6), gap_vs_real=round(f1 - ref, 6),
                       seconds=round(time.time() - t0, 1))
            pd.DataFrame([row]).to_csv(SEARCH_FILE, mode="a", index=False,
                                       header=not os.path.isfile(SEARCH_FILE))
            print(f"{arm:7s} {i}/{len(cands)} {key:22s} macro F1 {f1:.4f}")

    rec = pd.read_csv(SEARCH_FILE)
    best = {}
    for arm in ARMS:
        a = rec[rec.arm == arm].sort_values("tstr_f1", ascending=False).iloc[0]
        best[arm] = dict(batch=int(a["batch"]), lr=float(a["lr"]), noise_dim=int(a["noise_dim"]))
    with open(SELECTED_FILE, "w") as fh:
        json.dump(best, fh, indent=2)
    print("selected:", best)


run_search()

## Selected configuration and spread over the eight candidates (Table 5)

In [ ]:
if os.path.isfile(SEARCH_FILE):
    rec = pd.read_csv(SEARCH_FILE)
    display(rec.sort_values(["arm", "tstr_f1"], ascending=[True, False])
               [["arm", "trial", "tstr_f1", "real_reference_f1"]].round(4))
    spread = rec.groupby("arm")["tstr_f1"].agg(["max", "min"])
    spread["spread over 8"] = spread["max"] - spread["min"]
    display(spread.round(4))

In [ ]:
if IN_COLAB and not str(CHECKPOINT_DIR).startswith("/content/drive"):
    for path in (SEARCH_FILE, SELECTED_FILE):
        if os.path.isfile(path):
            files.download(path)